# Inference: HFACS Extract & Classify — 41 Sub-Factors with fine-tuned Llama-3.1-8B-Instruct (LoRA)

This notebook loads the LoRA adapter trained in `lora_finetune_llama3_subfactors_extract_classify.ipynb` and runs it on a CSV of ASRS narratives, producing all 41 individual HFACS contributing-factor indicators plus `Q1_Error` / `Q2_Violation` / `Final_Class` for each row.

## 1. Before you start

1. **Runtime**: `Runtime > Change runtime type > A100 GPU`.
2. **Adapter already trained**: this notebook loads the adapter saved at `ADAPTER_DIR` (set below) from `lora_finetune_llama3_subfactors_extract_classify.ipynb` — no retraining happens here.
3. **Upload the CSV to classify** to Drive, e.g.:
   ```
   MyDrive/llm_hfcas_preconditions/data/llm_feature_inference_input_50_per_class.csv
   ```
   It must contain a narrative column (default: `narrative`).
4. **Heads up on runtime**: generation is unbatched and each answer has 44 JSON fields (`max_new_tokens=900`), so this is slower per row than the 8-category notebook — budget roughly 10-25 sec/row on an A100. For the 200-row RF file with `LIMIT = None`, expect roughly 30-80 minutes.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
!pip install -q -U transformers accelerate peft datasets huggingface_hub torchao

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# --- Config ---
DRIVE_DIR = "/content/drive/MyDrive/llm_hfcas_preconditions"

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_DIR = f"{DRIVE_DIR}/outputs/llama3.1-8b-subfactors-extract-classify-lora"  # adapter from training notebook

INPUT_CSV = f"{DRIVE_DIR}/data/llm_feature_inference_input_50_per_class.csv"
NARRATIVE_COLUMN = "narrative"
RESULTS_CSV = f"{DRIVE_DIR}/outputs/llama3.1-8b-subfactors-extract-classify-lora/rf_llm_subfactor_features_50_per_class.csv"

LIMIT = None  # 200 rows total, set to a small number first to test
MAX_NEW_TOKENS = 900
BATCH_SIZE = 8  # narratives per generate() call

In [ ]:
from huggingface_hub import login
login()  # paste your HF read token when prompted

## 2. Load fine-tuned model (base + LoRA adapter)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map={"": 0})
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

## 3. Load narratives and build prompts

In [ ]:
import pandas as pd

df = pd.read_csv(INPUT_CSV, low_memory=False)
if LIMIT is not None:
    df = df.head(LIMIT).copy()

print(f"Rows to classify: {len(df)}")
print(df[NARRATIVE_COLUMN].iloc[0][:500])

## 4. Run inference

In [ ]:
import json
from tqdm import tqdm

tokenizer.padding_side = "left"

def build_prompt(narrative):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.replace("{narrative}", str(narrative))},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_batch(narratives):
    prompts = [build_prompt(n) for n in narratives]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(out[i][input_len:], skip_special_tokens=True) for i in range(len(narratives))]

OUTPUT_FIELDS = [
    'Contributing_Weather',
    'Contributing_Environment - Non Weather Related',
    'Anomaly_ATC Issue All Types',
    'Anomaly_Aircraft Equipment Problem Critical',
    'Anomaly_Aircraft Equipment Problem Less Severe',
    'Anomaly_Ground Event / Encounter Weather / Turbulence',
    'Anomaly_Inflight Event / Encounter Weather / Turbulence',
    'Anomaly_Inflight Event / Encounter Wake Vortex Encounter',
    'Anomaly_Inflight Event / Encounter Bird / Animal',
    'Anomaly_Ground Event / Encounter FOD',
    'Anomaly_Ground Event / Encounter Vehicle',
    'Anomaly_Ground Event / Encounter Object',
    'Anomaly_Ground Event / Encounter Person / Animal / Bird',
    'Anomaly_Ground Event / Encounter Other / Unknown',
    'Anomaly_Inflight Event / Encounter Other / Unknown',
    'Anomaly_Flight Deck / Cabin / Aircraft Event Passenger Electronic Device',
    'Anomaly_Flight Deck / Cabin / Aircraft Event Smoke / Fire / Fumes / Odor',
    'HumanFactors_Communication Breakdown',
    'HumanFactors_Troubleshooting',
    'HumanFactors_Human-Machine Interface',
    'HumanFactors_Confusion',
    'HumanFactors_Distraction',
    'HumanFactors_Situational Awareness',
    'HumanFactors_Physiological - Other',
    'HumanFactors_Other / Unknown',
    'Contributing_Staffing',
    'Contributing_Manuals',
    'HumanFactors_Training / Qualification',
    'Contributing_MEL',
    'HumanFactors_Time Pressure',
    'Contributing_Company Policy',
    'Contributing_Procedure',
    'Contributing_Software and Automation',
    'Contributing_Equipment / Tooling',
    'Contributing_ATC Equipment / Nav Facility / Buildings',
    'Contributing_Chart Or Publication',
    'Contributing_Logbook Entry',
    'Contributing_Incorrect / Not Installed / Unavailable Part',
    'Contributing_Aircraft',
    'Contributing_Airport',
    'Contributing_Airspace Structure',
    'Q1_Error',
    'Q2_Violation',
    'Final_Class',
]

narratives = df[NARRATIVE_COLUMN].tolist()
records = []
for i in tqdm(range(0, len(narratives), BATCH_SIZE)):
    batch = narratives[i:i + BATCH_SIZE]
    gen_texts = generate_batch(batch)
    for gen_text in gen_texts:
        try:
            pred = json.loads(gen_text)
        except json.JSONDecodeError:
            pred = {}
        record = {field: pred.get(field, "INVALID") for field in OUTPUT_FIELDS}
        record["raw_output"] = gen_text
        records.append(record)

## 5. Save results

In [ ]:
results_df = pd.concat([df.reset_index(drop=True), pd.DataFrame(records)], axis=1)
results_df.to_csv(RESULTS_CSV, index=False)
print(f"Saved {len(results_df)} classified rows to: {RESULTS_CSV}")
results_df[["Final_Class", "Q1_Error", "Q2_Violation"]].head()